# NeuroGraph · Phase 7 · QLoRA fine-tune of the university assistant
Settings: Accelerator **GPU T4 x2**, Internet **On**. Input: the **neurograph-60-intake** notebook output (redacted university chunks). Secrets (**Add-ons → Secrets**, attached): `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`.

## 1. Protect preinstalled packages

In [ ]:
from importlib.metadata import version

def write_constraints(pkgs: list[str], path: str) -> None:
    with open(path, "w") as f:
        f.write("\n".join(f"{p}=={version(p)}" for p in pkgs))

write_constraints(["torch", "numpy", "scipy", "scikit-learn", "mne"], "/tmp/keep.txt")
print(open("/tmp/keep.txt").read())

## 2. Get the latest code + install (adds peft + bitsandbytes for 4-bit training)

In [ ]:
import os
R = "/kaggle/working/neurograph"
if not os.path.isdir(R):
    !git clone -q https://github.com/stuBBornButterfly/neurograph.git {R}
!git -C {R} pull -q
!git -C {R} log --oneline -1
%cd {R}
!pip install -q -r requirements.txt -c /tmp/keep.txt
!pip install -q -r requirements-finetune.txt -c /tmp/keep.txt
!pip install -q -e . -c /tmp/keep.txt

## 3. AuraDB credentials from Kaggle Secrets (never printed) + notebook 60 output into the working folder

In [ ]:
import glob, os, shutil
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for key in ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"]:
    os.environ[key] = secrets.get_secret(key)
src = os.path.dirname(glob.glob("/kaggle/input/**/institutional/chunks.jsonl", recursive=True)[0])
shutil.copytree(src, "/kaggle/working/derivatives/institutional", dirs_exist_ok=True)
print("loaded:", [k for k in ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"] if os.environ.get(k)], "| chunks from", src)

## 4. Step 7.1: Q&A pairs from the university chunks (Qwen2.5-7B teacher, NLI filter, held-out chunks)

In [ ]:
!cd /kaggle/working/neurograph && python -m scripts.make_qa_dataset 2>&1 | grep -v -i -e warn -e it/s -e Loading

## 5. Step 7.2: QLoRA training (4-bit Qwen2.5-3B + LoRA adapters)

In [ ]:
!cd /kaggle/working/neurograph && python -m scripts.train_qlora 2>&1 | grep -v -i -e warn -e it/s -e Loading

## 6. Step 7.3: base vs fine-tuned on held-out chunks + model card

In [ ]:
!cd /kaggle/working/neurograph && python -m scripts.eval_qlora 2>&1 | grep -v -i -e warn -e it/s -e Loading

## 7. The whole agent with the fine-tuned model (same 24 questions as notebook 60): adapter on every route, then on the policy route only (the one it was trained on)

In [ ]:
!cd /kaggle/working/neurograph && python -m scripts.run_agent_eval --adapter /kaggle/working/derivatives/qlora/adapter --tag _qlora 2>&1 | grep -v -i -e warn -e it/s -e Loading
!cd /kaggle/working/neurograph && python -m scripts.run_agent_eval --adapter /kaggle/working/derivatives/qlora/adapter --adapter-policy-only --tag _qlora_policy 2>&1 | grep -v -i -e warn -e it/s -e Loading

## 8. Step 7.4: merge the adapter into the base + Q4_K_M GGUF for the CPU demo (llama.cpp)
Optional upload to a Hugging Face model repo: add the secrets `HF_TOKEN` (write token) and `HF_REPO` (e.g. `<user>/neurograph-qwen2.5-3b-gguf`).

In [ ]:
for key in ["HF_TOKEN", "HF_REPO"]:  # optional; without them the GGUF stays in the notebook output
    try:
        os.environ[key] = secrets.get_secret(key)
    except Exception:
        pass
!cd /kaggle/working/neurograph && python -m scripts.export_gguf 2>&1 | grep -v -i -e warn -e it/s -e Loading